# Homework Assignment 4

**Deadline:** 27 April 2026
<br><br>
**Student names:**: A. Agung Prawira Negara & Nand Van Deun
<br>
**Total hours spend:** 45 h
<br><br>
**Important Remarks:**
<br>
* **Submitting the homework is a compulsory course component.** 
* Making mistakes or not always being able to completely solve an exercise is fine. The goal is to let you interactively explore the concepts from the lecture and identify common problems/misconceptions. If you could not solve/finish a task, please discuss why and what you tried using markdown cells.
* If you should have any questions please don't hesitate to ask them. This especially includes any questions on (astro)physical aspects of the exercises. You can use the discussions section in our Toledo Ultra course. Please do not give away your solution to problems to other student groups when using the discussions.
* Document what you are doing in markdown cells. Adding formulas is a good way to make your implementations in code easier to understand.
* Plots have to follow scientific standards. This includes the requirement of axis labels including measurement units if applicable!
* Indicate how much time you spend in total on this assignment.

**Submission Process:**
<br>
* Groups only need to submit a single notebook, there is no need for individual submissions.
* Rename the notebook as `HomeworkAssignment4_FirstnameLastname1_FirstnameLastname2.ipynb` where you insert your names in the filename.
* Send your notebook to both TAs (Dario Fritzewski: dario.fritzewski@kuleuven.be; Reinhold Willcox: reinhold.willcox@kuleuven.be) using the subject line "**Data Analysis Homework Submission 4**". You can make your notebook file smaller by deleting the outputs before submitting. If the notebook should still be too large for an email, please use [WeTransfer](https://wetransfer.com/).

⚠️ <font color='red' style='bold'><b>IMPORTANT:</b></font> You should solve at least one task analytically, i.e. without the use of sampling-based methods. We recommend to always try to come up with an analytical approach and optionally provide an additional MCMC based solution.

# Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import pymc as pm


%matplotlib inline
from matplotlib import pyplot as plt
from IPython.display import Image

# Feel free to change these defaults
plt.rc('font',   size=16)          # controls default text sizes
plt.rc('axes',   titlesize=18)     # fontsize of the axes title
plt.rc('axes',   labelsize=18)     # fontsize of the x and y labels
plt.rc('xtick',  labelsize=18)     # fontsize of the tick labels
plt.rc('ytick',  labelsize=18)     # fontsize of the tick labels
plt.rc('legend', fontsize=18)      # legend fontsize
plt.rc('figure', titlesize=18)     # fontsize of the figure title
plt.rc('figure', dpi=75)           # Changed to 100 for some version >3.5.2

# Contents

1. [A trip to the casino](#Task-1:-A-trip-to-the-casino)
2. [Solar flares](#Task-2:-Solar-flares)
3. [Supernova neutrinos](#Task-3:-Supernova-neutrinos)

For the credits of the images, see [the homework page](https://github.com/JorisDeRidder/DataAnalysisInPhysicsAndAstronomy/blob/main/Images/credits.txt) of this course on Github.

---

## Task 1: A trip to the casino

The Gambler's fallacy refers to the erroneous belief that statistically independent events are affected by previous repeated outcomes. For instance, that after repeated rolls of black in a roulette, the following rolls are more likely to result in a red.

In [ ]:
url = "https://raw.githubusercontent.com/JorisDeRidder/DataAnalysisInPhysicsAndAstronomy/main/Images/Roulette_casino.JPG"
Image(url, width=600)

One could try to argue with a gambler friend that this is not true using synthetic data, but he might argue back that our random numbers do not reproduce all the atoms of a roulette where the memory of previous rolls is stored. So let us test this using real data from a casino. Most casinos actually keep this data private, but the Wiesbaden Casino in Germany provides information on each of their tables for every day (https://www.spielbank-wiesbaden.de/index.php?id=101). The dataset we will use is a compilation of data for all days in 2018 for Table 1 in the casino. The roulette is of European style, with numbers from 1 to 36 evenly distributed between black and red, and a single green zero.

In [ ]:
url = "https://raw.githubusercontent.com/JorisDeRidder/DataAnalysisInPhysicsAndAstronomy/main/Datasets/wiesbaden_table1.txt"
data = pd.read_csv(url)

In [ ]:
data.head()

The columns in the dataset are:
- **date**: the date at which the roll was made (yyyymmdd format)
- **roll**: the number of the roll of the table in the year (data is ordered in time)
- **color**: the color of the roll, with "R" being red, "B" being black and "G" being green
- **number**: the number that was drawn

First we will test if the roulette is fair and then test if repeated draws on this roulette affect the subsequent outcome. Your tasks for this exercise are the following:
- Using a Bayesian analysis, determine the posterior distribution for the probability $\theta$ of drawing a red number.  Use a flat prior for $\theta$ (i.e. $P(\theta)\propto 1$) and evaluate the posterior using the first month of data, the first six months, and the full year. To quantify the uncertainty in the result, consider a 99.73\% probability interval (equivalent to 3-$\sigma$ for a normal distribution) of the posterior by computing the $0.27/2$ and $100-0.27/2$ percentiles of the posterior samples. Within this probability interval, do you find that the roulette is fair? Is this outcome affected if you use a Jeffreys prior $P(\theta)\propto \theta^{-1/2}(1-\theta)^{-1/2}$?
- Using the full dataset, determine the posterior distribution to draw each of the numbers (0-36) in the roulette. Within the 99.73\% probability interval of the posteriors, is any outcome inconsistent with the expected probability? What about within a 95\% probability interval (2-$\sigma$ equivalent)? If you find a mismatch, is it statistically meaningful?
- Now, let us test if repeated draws affect the subsequent result. Any 'good' gambler will tell you that a table loses its memory from one day to the next, so restrict your analysis to groups of draws from individual days. Check the number of red draws happening after two subsequent red draws. For this set, determine the posterior probability $\theta'$ to draw a red ball after 2 previous red balls, and compare this to the posterior probability derived from all the data in the first part of this exercise. Within the 99.73\% probability interval of the posterior, do you see a mismatch?

### Sub-task A: Posterior for P(red)
We use roulette data from Table 1 of the Wiesbaden Casino for the full year 2018.
Each row corresponds to a single spin, recording the date, roll number, color (R/B/G),
and number drawn. We first load the data and inspect its structure.

In [ ]:
print(data.head(10))
print(f"\nTotal spins: {len(data)}")
print(f"Color counts:\n{data['color'].value_counts()}")
print(f"\nDate range: {data['date'].min()} to {data['date'].max()}")
print(f"Unique dates: {data['date'].nunique()}")

The dataset contains 51,419 spins across 269 days, with slightly more black (25,292)
than red (24,762) outcomes. We now compute the posterior for θ for three time periods
which are first month, first six months, and full year in order to observe how the posterior evolves
as more data is accumulated.

In [ ]:
# Filter datasets
data_1m  = data[data['date'] <= 20180131]
data_6m  = data[data['date'] <= 20180630]
data_full = data.copy()

def get_posterior(subset, prior=(1, 1)):
    """
    Returns a scipy Beta distribution representing the posterior.
    prior: (alpha0, beta0) — (1,1) for flat, (0.5, 0.5) for Jeffreys
    """
    k = (subset['color'] == 'R').sum()   # number of reds
    N = len(subset)                       # total spins
    alpha = k + prior[0]
    beta  = (N - k) + prior[1]
    return stats.beta(alpha, beta), k, N

# Flat prior posteriors
post_1m_flat,   k1,  N1  = get_posterior(data_1m)
post_6m_flat,   k6,  N6  = get_posterior(data_6m)
post_full_flat, kf,  Nf  = get_posterior(data_full)

# Jeffreys prior posteriors
post_1m_jeff,   _,   _   = get_posterior(data_1m,   prior=(0.5, 0.5))
post_6m_jeff,   _,   _   = get_posterior(data_6m,   prior=(0.5, 0.5))
post_full_jeff, _,   _   = get_posterior(data_full, prior=(0.5, 0.5))

print(f"1 month:  k={k1}, N={N1}")
print(f"6 months: k={k6}, N={N6}")
print(f"Full year: k={kf}, N={Nf}")

We plot the posterior distributions for both priors across the three time periods,
and mark the 99.73% probability interval and the fair value θ_fair = 18/37.

In [ ]:
theta = np.linspace(0.45, 0.52, 2000)
fair_theta = 18/37  # expected for a fair roulette

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

datasets = [
    (post_1m_flat,   post_1m_jeff,   "1 month",   N1),
    (post_6m_flat,   post_6m_jeff,   "6 months",  N6),
    (post_full_flat, post_full_jeff, "Full year", Nf),
]

for ax, (post_flat, post_jeff, label, N) in zip(axes, datasets):
    pdf_flat = post_flat.pdf(theta)
    pdf_jeff = post_jeff.pdf(theta)

    ax.plot(theta, pdf_flat, label="Flat prior",     color="steelblue")
    ax.plot(theta, pdf_jeff, label="Jeffreys prior", color="tomato", linestyle="--")
    ax.axvline(fair_theta, color="black", linestyle=":", label=f"Fair: {fair_theta:.4f}")

    # 99.73% interval for flat prior
    lo, hi = post_flat.ppf(0.0027/2), post_flat.ppf(1 - 0.0027/2)
    ax.axvspan(lo, hi, alpha=0.15, color="steelblue", label=f"99.73% CI [{lo:.4f}, {hi:.4f}]")

    ax.set_title(f"{label} (N={N})")
    ax.set_xlabel(r"$\theta$ = P(red)")
    ax.set_ylabel(r"$P(\theta \mid \mathrm{data})$")
    ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig("task1a_posteriors.png", dpi=150)
plt.show()

# Print summary table
print(f"{'Dataset':<12} {'k':>6} {'N':>6} {'Mean':>8} {'99.73% lo':>10} {'99.73% hi':>10} {'Fair inside?':>13}")
print("-" * 65)
for post_flat, label, k, N in [
    (post_1m_flat,   "1 month",   k1, N1),
    (post_6m_flat,   "6 months",  k6, N6),
    (post_full_flat, "Full year", kf, Nf),
]:
    lo, hi = post_flat.ppf(0.0027/2), post_flat.ppf(1 - 0.0027/2)
    inside = "YES" if lo <= fair_theta <= hi else "NO"
    print(f"{label:<12} {k:>6} {N:>6} {post_flat.mean():>8.5f} {lo:>10.5f} {hi:>10.5f} {inside:>13}")

Both priors yield virtually identical posteriors, with N ~ 10³–10⁴ the likelihood
dominates and the prior choice becomes irrelevant. The posterior narrows progressively
with more data, reflecting reduced uncertainty. In all three cases, θ_fair = 18/37
falls within the 99.73% interval, so there is no evidence that the roulette is biased.

### Sub-task B: Posterior for each number (0–36)

We apply the same approach to each number i, with θ_i = P(number i) and
posterior Beta(k_i+1, N-k_i+1) under a flat prior. A fair roulette gives
θ_i = 1/37 ≈ 0.0270. We check consistency at the 99.73% and 95% levels.

In [ ]:
numbers = np.arange(0, 37)
fair_p = 1/37
N_full = len(data_full)

low_p,  high_p  = 0.0027/2, 1 - 0.0027/2   # 99.73%
low_p2, high_p2 = 0.05/2,   1 - 0.05/2      # 95%

results = []
for num in numbers:
    k = (data_full['number'] == num).sum()
    post = stats.beta(k + 1, N_full - k + 1)
    lo_3s, hi_3s = post.ppf(low_p),  post.ppf(high_p)
    lo_2s, hi_2s = post.ppf(low_p2), post.ppf(high_p2)
    results.append({
        'number': num,
        'k': k,
        'mean': post.mean(),
        'lo_3s': lo_3s, 'hi_3s': hi_3s,
        'lo_2s': lo_2s, 'hi_2s': hi_2s,
        'outside_3s': not (lo_3s <= fair_p <= hi_3s),
        'outside_2s': not (lo_2s <= fair_p <= hi_2s),
    })

results = pd.DataFrame(results)
print(results[['number','k','mean','lo_3s','hi_3s','outside_3s','outside_2s']].to_string(index=False))

We visualise the posterior means and probability intervals for all 37 numbers.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.errorbar(numbers, results['mean'], 
            yerr=[results['mean'] - results['lo_3s'], results['hi_3s'] - results['mean']],
            fmt='o', color='steelblue', ecolor='steelblue', alpha=0.5,
            capsize=3, label='99.73% interval')

ax.errorbar(numbers, results['mean'],
            yerr=[results['mean'] - results['lo_2s'], results['hi_2s'] - results['mean']],
            fmt='o', color='tomato', ecolor='tomato', alpha=0.8,
            capsize=3, label='95% interval')

ax.axhline(fair_p, color='black', linestyle='--', label=f'Fair: 1/37 = {fair_p:.4f}')

ax.set_xlabel('Number')
ax.set_ylabel('P(number)')
ax.set_title('Posterior mean and probability intervals for each roulette number')
ax.set_xticks(numbers)
ax.legend()
plt.tight_layout()

At the 99.73% level, no number is inconsistent with 1/37, no evidence of bias.
At the 95% level, numbers 7 and 20 fall outside their interval. However, testing
37 numbers simultaneously, we expect ~37 × 0.05 ≈ 1.85 spurious excursions by
chance alone, so finding 2 is not statistically meaningful.

### Sub-task C: Testing for memory

We select draws immediately following two consecutive reds, restricted within each
day. For this subset (N' draws, k' red), the posterior for θ' = P(red | prev. two red) is:

$$P(\theta' \mid k', N') = \text{Beta}(k'+1,\ N'-k'+1)$$

We compare the 99.73% intervals of θ and θ' to test for a memory effect.

In [ ]:
# Work day by day to avoid cross-day contamination
mask = []
for date, group in data_full.groupby('date'):
    colors = group['color'].values
    for i in range(len(colors)):
        if i >= 2 and colors[i-1] == 'R' and colors[i-2] == 'R':
            mask.append(True)
        else:
            mask.append(False)

data_full['after_two_reds'] = mask

subset_c = data_full[data_full['after_two_reds']]
print(f"Draws after two consecutive reds: {len(subset_c)}")
print(f"Red among those: {(subset_c['color'] == 'R').sum()}")

We find 11,803 draws following two consecutive reds, of which 5,642 are red.
We now compute the posterior for θ' and compare it directly to the full-year
posterior for θ from sub-task A.

In [ ]:
# Posterior for theta' (red after two consecutive reds)
k_c = (subset_c['color'] == 'R').sum()
N_c = len(subset_c)
post_c = stats.beta(k_c + 1, N_c - k_c + 1)

# Compare with full-year posterior from sub-part A
theta = np.linspace(0.455, 0.515, 2000)

lo_c,  hi_c  = post_c.ppf(0.0027/2),       post_c.ppf(1 - 0.0027/2)
lo_f,  hi_f  = post_full_flat.ppf(0.0027/2), post_full_flat.ppf(1 - 0.0027/2)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(theta, post_full_flat.pdf(theta), color='steelblue', label=f'All draws (θ, N={Nf})')
ax.plot(theta, post_c.pdf(theta),         color='tomato',    label=f'After two reds (θ\', N={N_c})')
ax.axvline(fair_theta, color='black', linestyle=':', label=f'Fair: {fair_theta:.4f}')
ax.axvspan(lo_f, hi_f, alpha=0.15, color='steelblue')
ax.axvspan(lo_c, hi_c, alpha=0.15, color='tomato')

ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$P(\theta \mid \mathrm{data})$")
ax.set_title("Posterior comparison: all draws vs after two consecutive reds")
ax.legend()
plt.tight_layout()


In [ ]:
print(f"θ  (all draws):        mean={post_full_flat.mean():.5f}, 99.73% CI [{lo_f:.5f}, {hi_f:.5f}]")
print(f"θ' (after two reds):   mean={post_c.mean():.5f}, 99.73% CI [{lo_c:.5f}, {hi_c:.5f}]")
print(f"\nDo the 99.73% intervals overlap? ", 
      "YES" if lo_c <= hi_f and lo_f <= hi_c else "NO")

The 99.73% intervals of θ and θ' overlap substantially, and the posterior means
differ by less than 0.004. There is no evidence of a memory effect, the probability
of drawing red is statistically indistinguishable regardless of previous outcomes.
The data does not support the gambler's fallacy.

---

## Task 2: Solar flares

Solar flares are sudden changes in brightness in the surface of the Sun that can be observed in X-rays, and are associated with interactions of the tangled magnetic fields in its surface. In the most extreme cases, large flares can be connected to coronal mass ejections, large blobs of plasma that can travel the solar system at thousands of kilometers per second and cause significant damage if we are unlucky to cross their path. One extreme case was the one observed by the English Solar astrophysicist Richard Carrington on September 1st, 1859. On that date, Carrington observed a large group of sunspots grouped together which suddenly emmited a bright flash of light. The following day, auroras we're visible all over the world, some so bright that they were visible during the day. Despite the beautiful display, the event had a large impact on telegraph networks, and were it to happen today it could result in catastrophic damages to our communication infrastructure and electric grids.

In [ ]:
url = "https://raw.githubusercontent.com/JorisDeRidder/DataAnalysisInPhysicsAndAstronomy/main/Images/Watching_the_Sun_SOHO_pillars.jpg"
Image(url, width=700)

In this exercise, we will explore a database of solar flares detected by the GOES satellite. GOES is a geostationary satellite that can detect solar flares with its onboard X-ray detector, and the dataset in question was produced using the method of [Ryan et al. (2012)](https://ui.adsabs.harvard.edu/abs/2012ApJS..202...11R/abstract) to properly account for the biases of the instrument. Depending on the maximum energy flux reached by an event, flares are categorized (in order of increasing strength) in classes A, B, C, M and X. The provided dataset is simplified, but if you're interested in it you can download it in full [here](https://data.nas.nasa.gov/helio/portals/solarflares/dataproducts.html).

In [ ]:
url = "https://raw.githubusercontent.com/JorisDeRidder/DataAnalysisInPhysicsAndAstronomy/main/Datasets/GOES_flare_data.txt"
data = pd.read_csv(url, sep='\s+')

In [ ]:
data.head()

The columns in the dataset are:
- **start_day**: the date at which the flare was first detected (yyyymmdd format)
- **start_time**: the time (UT) at which the flare was first detected
- **peak_day**, **peak_time**, **end_day**, **end_time**: Same as the previous two but for the moment of peak energy flux detected by the satellite and for the moment the flare stopped being detected.
- **GOES_class**: classification of the flare according to the energy flux measured by the satellite. Full data provides subcategories beyond the A, B, C, M and X classes, but we just provide the general class of the event here.

For this exercise, we really just need the **start_day** and **GOES_class** data columns.

Your tasks for this exercise are the following:
- First, we will estimate the number of flares per year. For this, we take the yearly number of flares $n$ and model it as a Poisson process with a distribution $f(n,\lambda)=e^{-\lambda}\lambda^{n}/n!$, where $\lambda$ is the expected number of events per year. For each year, compute the posterior of $\lambda$ using the Jeffrey's prior for a Poisson distribution ($p(\lambda)\propto \lambda^{-1/2}$) and determine its median and 99.73% probability interval. Plot your results and discuss if you can state that the rate evolves with time.
- Look for information on the solar cycle and the count of solar spots. Does this match your results from the previous step?
- Can we determine if the relative fraction of different flares changes with time? Consider flares of each type as determined by the **GOES_class** column and for each year compute the fraction of each type to the total number of flares that year. For simplicity, model the fraction of each particular flare independently with a binomial distribution likelihood, and determine posterior distributions, their median and their 99.73% probability intervals (make your own choice on the prior). Discuss whether the data is sufficient to determine if these fractions evolve with time. Note that flares of class A are at the detection limit of the instrument.

In [ ]:
data["year"] = data["start_day"].astype(str).str.slice(0, 4).astype(int)
year_counts = data.groupby("year").size()
years = year_counts.index.values
yearly = year_counts.values
print(yearly)


In [ ]:
medians = []
lowers = []
uppers = []

for k in yearly:
    with pm.Model() as myModel:

        #xobs = pm.MutableData("xobs", x)
        yobs = yearly #pm.MutableData("yobs", yearly)

        lam = pm.Gamma("lam", alpha=0.5, beta=1e-6)

        likelihood = pm.Poisson("y", mu=lam, observed=k)

        trace = pm.sample(3000, chains=4, cores=2, return_inferencedata=True)

    samples = trace.posterior["lam"].values.flatten()
    medians.append(np.median(samples))
    lowers.append(np.quantile(samples, 0.00135))
    uppers.append(np.quantile(samples, 0.99865))

In [ ]:
years = np.arange(2002, 2020, 1)

plt.figure(figsize=(16, 6))

plt.plot(years, medians, marker="o", label="Posterior median λ")
plt.fill_between(years, lowers, uppers, alpha=0.2, label="99.73% credible interval")

plt.xlabel("Year")
plt.xticks(np.arange(2002, 2020, 1))
plt.ylabel("Expected number of flares per year (λ)")
plt.title("Posterior Poisson rate per year (Jeffreys prior)")
plt.legend()
plt.show()

Looking at the posterior expected number of flares, we see an oscillating behaviour, with the maximum of the peaks decreasing. This does match up with the solar cycles.

In [ ]:
# total flares per year
year_totals = data.groupby("year").size().rename("n")

# counts per year and GOES class
year_class_counts = data.groupby(["year", "GOES_class"]).size().rename("k").reset_index()

In [ ]:
classes = sorted(data["GOES_class"].unique())
results = []

for c in classes:
    df_c = year_class_counts[year_class_counts["GOES_class"] == c]

    for _, row in df_c.iterrows():
        year = row["year"]
        k = row["k"]
        n = year_totals.loc[year]

        with pm.Model():
            p = pm.Beta("p", alpha=0.5, beta=0.5)  # Jeffreys prior
            k_obs = pm.Binomial("k_obs", n=n, p=p, observed=k)
            trace = pm.sample(2000, tune=2000, chains=2, progressbar=False)

        samples = trace.posterior["p"].values.flatten()

        median = np.median(samples)
        lower = np.quantile(samples, 0.00135)
        upper = np.quantile(samples, 0.99865)

        results.append((year, c, n, k, median, lower, upper))

res_df = pd.DataFrame(
    results,
    columns=["year", "class", "n", "k", "median", "lower", "upper"]
)

In [ ]:
for c in classes:
    sub = res_df[res_df["class"] == c].sort_values("year")

    plt.figure(figsize=(10, 4))
    plt.plot(sub["year"], sub["median"], marker="o", label=f"{c}-class median fraction")
    plt.fill_between(sub["year"], sub["lower"], sub["upper"], alpha=0.2,
                     label="99.73% credible interval")
    plt.ylim(0, 1)
    plt.xlabel("Year")
    plt.xticks(np.arange(2002, 2021, 2))
    plt.ylabel(f"Fraction of {c}-class flares")
    plt.title(f"Posterior fraction of {c}-class flares over time")
    plt.legend()
    plt.tight_layout()
    plt.show()

We can see that both B and C class fractions oscillate in the opposite direction, again corresponding to the solar cycle, For the M class as well but less significant. Far A and X class it is not sufficient

---

## Task 3: Supernova neutrinos

When a massive star dies, it becomes a supernova where the core of the star completely collapses. Depending on the mass of the star, the result of this collapse can be the formation of a neutron star, an extremely dense object that can contain around one or two times the mass of the Sun in a radius of about 10 kilometers. At these high densities, most of the protons from which the neutron star is formed undergo an electron capture process, where a proton and an electron combine to form a neutron and a neutrino. From this process, a supernova is not just expected to be associated with a luminous explosion, but also with a burst of neutrinos.

This was one of the motivations behind the construction of neutrino detectors, which potentially allow for the simultaneous detection of an astrophysical event both with light as well as with neutrinos. And that is exactly what was observed in 1987, when a supernova was detected in the Large Magellanic Cloud. Given the proximity of this supernova to us ("only" 168,000 lightyears) the event was recorded in extreme detail, and current observations can see in real time how the ejected material interacts with its surroundings (see the movie below). More important, in the context of this exercise, is that the supernova was close enough to Earth that its neutrino burst (lasting just a few seconds) could be measured in two detectors: the IMB detector in the US and the Kamiokande detector in Japan. This is regarded as the first example of multi-messenger (light as well as neutrinos) astronomy from a source outside the solar system.

![1987a](https://raw.githubusercontent.com/JorisDeRidder/DataAnalysisInPhysicsAndAstronomy/main/Images/1987a.gif "1987a")

In [ ]:
url = "https://raw.githubusercontent.com/JorisDeRidder/DataAnalysisInPhysicsAndAstronomy/main/Images/IMB_tank.png"
Image(url, width=1000)

# A picture of the inside of the Irvine-Michigan-Brookhaven (IMB) detector. This Cherenkov detector was built in the Morton salt mine 
# in Ohio and consist of a massive tank of more than 9 million liters of ultra-pure water, and a large network of phototubes attached 
# to the walls. The neutrinos show up by way of their interactions with protons in the water.

The objective of this exercise is to evaluate the significance of the observation made by the IMB detector. The measurement is discussed in [Bionta et al. (1987)](https://ui.adsabs.harvard.edu/abs/1987PhRvL..58.1494B/abstract), where they report the search for a burst in a 6.4 hour period of data from the detector. Separating the data in intervals of 10 seconds, they identified one interval during which 9 events where detected. As the detector is sensitive as well to neutrinos produced by the sun and cosmic rays, it is important to validate that these 9 detections are significant. The provided dataset contains the number of 10-second intervals for which a specific number of events was detected during the aforementioned 6.4 hour period.

In [ ]:
data = pd.DataFrame({"events":[0,1,2,3,4,5,6,7,8,9], "number":[1043,860,307,78,15,3,0,0,0,1]})
data

The columns in the dataset are:
- **events**: number of events detected in a 10-second interval
- **number**: number of 10 second intervals with the corresponding number of detected events.

Your tasks for this exercise are to use Bayesian analysis (making your own decision on the priors to use) to answer the following:

- Assuming a Poisson distribution for the events, and ignoring the time interval with 9 events which is the supposed neutrino burst, determine a posterior distribution for the rate of events $\lambda$ using the total number of events detected.
- Since we do not only have the number of events detected in the 6.4 hour period but also the distribution at which specific numbers of events happened in 10-second intervals, we can perform a Bayesian analysis based on the probability of obtaining this precise distribution. The distribution function that describes this is the multinomial distribution, a generalization of the binomial distribution function that can be used to model loaded dice, for example. Look up the form of this distribution.<br><br>
To model the process with a multinomial distribution, denote by $k_i$ the number of 10-second intervals out of a total of $n$ intervals for which $i$ events were detected, and by $k_{\geq 6}$ the number of 10-second intervals for which 6 or more events were detected (which is $k_{\geq 6}=0$ if we exclude the supposed neutrino burst). The probability of drawing exactly a specific set of $\{k_0,k_1,k_2,k_3,k_4,k_5,k_{\geq 6}\}$ for a given value of the rate $\lambda$ is then given by a multinomial distribution function with 7 outcomes and $n$ draws, where the probability of each outcome can be computed from a Poisson distribution. Use this probability distribution as the likelihood in a Bayesian analysis to determine a posterior distribution for $\lambda$, Using the same prior as in the previous part of the exercise.<br><br>
<font color='red'>**Warning:** </font> evaluating the multinomial probability distribution can be tricky for large $n$ as parts of it can overflow the precision of integer numbers in your computer for large $n$. We recommend you use the [scipy implementation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.multinomial.html).
- Compare the two posterior distributions derived. Does the posterior that uses the full distribution of the events provide a better constraint on $\lambda$? Can you explain the result?
- Given your posteriors on the rate $\lambda$, estimate the probability that the 9 event interval was randomly produced given the rate estimate and the 6.4 hour interval for the search. Assuming the detection by the Japanese Kamiokande detector was of equal significance, discuss how this increases the likelihood of the event being a true anomaly associated with the supernova.

In [ ]:
def get_posterior(subset, prior=(1, 1)):
    """
    Returns a scipy Geta distribution representing the posterior.
    prior: (alpha0, beta0) —  (0.5, 1e-6) for Jeffreys
    """
    k = ((subset["events"]*subset["number"]))[0:-1].sum()   # number of events
    t = (subset["number"].sum()-1)*10                     # total time
    rate = k/t
    alpha = k + prior[0]
    beta  = t + prior[1]
    return stats.gamma(alpha, scale=1/beta), k, t, alpha, beta


print(get_posterior(data, (0.5, 1e-6)))
